In [195]:
import pandas as pd
import importlib.util
from pathlib import Path

# use the project's real CSV-backed random number generator (repo root), not a local reimplementation
_ROOT_RNG_PATH = r"E:\Coding Site\just_for_fun\random_no_generation.py"
_spec = importlib.util.spec_from_file_location("root_random_no_generation", _ROOT_RNG_PATH)
rng = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rng)

result_dir = Path("generated_result")

csv_files = sorted(result_dir.glob("*.csv"))
dfs = {f.stem: pd.read_csv(f) for f in csv_files}

In [196]:
df_alma_vale = dfs["df_results_alma_vale_pending_verification"]
df_almavale_referendum1 = dfs["df_results_almavale_referendum1_pending_verification"]
df_generaldirect_referendum1 = dfs["df_results_generaldirect_referendum1_pending_verification"]
df_13D = dfs["df_results_13D_pending_verification"]
df_general_proportional = dfs["df_results_general_proportional_pending_verification"]
df_general_direct = dfs["df_results_general_direct_pending_verification"]

df_alma_vale.head()

,Shape_ID,Electorate,Turnout,Left_Left,Left_Right,Left_MR,Right_Left,Right_Right,Right_MR,MR_Left,MR_Right,MR_MR,Spoil,received_at,verified_at,declared_at
0,A01_001,8063,5697,513,448,1184,133,191,357,1266,797,780,28,NaN,NaN,NaN
1,A01_002,7316,5148,984,597,1891,41,50,99,751,335,385,15,NaN,NaN,NaN
2,A01_003,3947,2806,352,261,531,90,181,218,471,366,317,19,NaN,NaN,NaN
3,A01_004,5603,4964,322,252,750,69,112,247,1384,797,1008,23,NaN,NaN,NaN
4,A01_005,3935,2967,317,271,659,60,100,165,664,372,343,16,NaN,NaN,NaN


## Validation

Column names differ per file (`Shape_ID` vs `Constituency`/`Sub_Constituency`, and different candidate/party columns like `Left_Left` vs `Yes_Votes`/`No_Votes` vs `NRD`/`LRP`/...). Rather than hardcoding vote-column names per file, we treat every column **not** in a fixed metadata set as a "vote column" for that file, and sum those (plus `Spoil`) to get total votes cast.

Checks run per row, per file:
1. `turnout_gt_electorate` - Turnout > Electorate (impossible, should never happen)
2. `total_gt_turnout` - (sum of vote columns + Spoil) > Turnout (more ballots counted than people who voted)
3. `total_ne_turnout` - (sum of vote columns + Spoil) != Turnout (every counted ballot should be accounted for exactly)

Check 3 subsumes check 2, but we keep them separate since an "exceeds" failure is a harder data-integrity error than a simple mismatch (which could be short by a few uncounted/pending ballots).

In [197]:
META_COLS = {
    "Shape_ID", "Constituency", "Sub_Constituency",
    "Electorate", "Turnout", "Spoil",
    "received_at", "verified_at", "declared_at",
}

def vote_columns(df):
    """Every column that isn't fixed metadata is treated as a vote/candidate column."""
    return [c for c in df.columns if c not in META_COLS]

def validate_results(df, name):
    df = df.copy()
    vcols = vote_columns(df)

    df["_votes_counted"] = df[vcols].sum(axis=1)
    df["_total_cast"] = df["_votes_counted"] + df["Spoil"]

    df["turnout_gt_electorate"] = df["Turnout"] > df["Electorate"]
    df["total_gt_turnout"] = df["_total_cast"] > df["Turnout"]
    df["total_ne_turnout"] = df["_total_cast"] != df["Turnout"]

    df["_source"] = name
    return df

validated = {name: validate_results(df, name) for name, df in dfs.items()}

In [198]:
summary_rows = []
for name, vdf in validated.items():
    summary_rows.append({
        "file": name,
        "rows": len(vdf),
        "turnout_gt_electorate": vdf["turnout_gt_electorate"].sum(),
        "total_gt_turnout": vdf["total_gt_turnout"].sum(),
        "total_ne_turnout": vdf["total_ne_turnout"].sum(),
    })

summary = pd.DataFrame(summary_rows).set_index("file")
summary

,rows,turnout_gt_electorate,total_gt_turnout,total_ne_turnout
file,,,,
df_results_13D_pending_verification,12,0,0,0
df_results_alma_vale_pending_verification,20850,0,0,0
df_results_almavale_referendum1_pending_verification,20850,0,0,0
df_results_general_direct_pending_verification,15,0,0,0
df_results_general_proportional_pending_verification,14,0,0,0
df_results_generaldirect_referendum1_pending_verification,15,0,0,0


In [199]:
# Inspect the failing rows across every file (any of the three checks tripped)
id_cols = ["Shape_ID", "Constituency", "Sub_Constituency"]
report_cols = ["_source", "Electorate", "Turnout", "Spoil", "_votes_counted", "_total_cast",
               "turnout_gt_electorate", "total_gt_turnout", "total_ne_turnout"]

failures = []
for name, vdf in validated.items():
    mask = vdf["turnout_gt_electorate"] | vdf["total_gt_turnout"] | vdf["total_ne_turnout"]
    present_id_cols = [c for c in id_cols if c in vdf.columns]
    failures.append(vdf.loc[mask, present_id_cols + report_cols])

all_failures = pd.concat(failures, ignore_index=True) if failures else pd.DataFrame()
print(f"{len(all_failures)} failing rows across {len(validated)} files")
all_failures.head(50)

0 failing rows across 6 files


,Constituency,Sub_Constituency,_source,Electorate,Turnout,Spoil,_votes_counted,_total_cast,turnout_gt_electorate,total_gt_turnout,total_ne_turnout,Shape_ID


## Timestamps

`received_at`, `verified_at`, `declared_at` need to be stamped on the now-validated rows. Timestamps must be UTC ISO-8601 in the form `YYYY-MM-DDTHH:MM:SSZ`, e.g. `2026-08-02T16:21:39Z`.

Edit the placeholder values below before running the rest of the notebook:
- `d13_received_time` / `d13_validated_time` -> applied to `df_results_13D_pending_verification`
- `general_received_time` / `general_validated_time` -> applied to the three `general_*` files (`general_direct`, `general_proportional`, `generaldirect_referendum1`), since they're counted together

(Renamed from `13d_...` since Python identifiers can't start with a digit.)

In [200]:
# EDIT THESE - format must be "YYYY-MM-DDTHH:MM:SSZ" (UTC, seconds precision, trailing "Z")
d13_received_time = "2026-08-11T02:00:00Z"
d13_validated_time = "2026-08-11T10:00:00Z"

general_received_time = "2026-08-11T02:00:00Z"
general_validated_time = "2026-08-11T12:30:00Z"

### df_results_13D: received / verified / declared

In [201]:
import random
import time

TS_FORMAT = "%Y-%m-%dT%H:%M:%SZ"
random.seed(time.time())  # seeded from current time, so declared_at differs on every run

def random_offset_time(base_str, min_hours, max_hours):
    base = pd.Timestamp(base_str)
    offset_seconds = random.uniform(min_hours * 3600, max_hours * 3600)
    return (base + pd.Timedelta(seconds=offset_seconds)).strftime(TS_FORMAT)

def declared_time_13D(electorate):
    if electorate <= 150_000_000:
        return random_offset_time(d13_validated_time, 3, 5)
    return random_offset_time(d13_validated_time, 5, 8)

df_13D["received_at"] = d13_received_time
df_13D["verified_at"] = d13_validated_time
df_13D["declared_at"] = df_13D["Electorate"].apply(declared_time_13D)

df_13D[["Constituency", "Sub_Constituency", "Electorate", "received_at", "verified_at", "declared_at"]]

,Constituency,Sub_Constituency,Electorate,received_at,verified_at,declared_at
0,Shek East and Central,Shek East and Central,675100000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T15:17:15Z
1,"Shek West, Rainbow N, LF & Huang",Shek West,393600000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T16:14:25Z
2,"Shek West, Rainbow N, LF & Huang",Mid and North Rainbow,228200000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T17:15:53Z
3,"Shek West, Rainbow N, LF & Huang",LF,42800000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T13:52:38Z
4,"Shek West, Rainbow N, LF & Huang",Huang,56500000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T14:17:59Z
5,Tong,Tong Central,14800000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T13:11:33Z
6,Tong,Mid Tong,57500000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T14:32:28Z
7,Tong,Upper Tong,8500000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T13:14:59Z
8,Diamond & Rainbow SE,Diamond,530000000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T15:57:41Z
9,Diamond & Rainbow SE,Rainbow Southwest,135000000,2026-08-11T02:00:00Z,2026-08-11T10:00:00Z,2026-08-11T14:07:50Z


### df_general_direct / df_general_proportional / df_generaldirect_referendum1

`received_at` / `verified_at` are stamped from `general_received_time` / `general_validated_time` on all three.

`declared_at` is a dependency cascade (confirmed rules):
- **Counting rate**: 6000 votes per 0.5h, linear -> 12,000 votes/hour.
- **Rounding**: counting time is rounded **up** to the next half hour (ceiling), so there's always enough time allotted.
- **Base counting time** = `extra_ballot_proceed_speed` (2.5) x that rounded half-hour figure.
- **Noise**: every individual counting time (`df_general_direct` non-Swift rows, and each of the 6 Swift tables in `df_general_proportional`) is sampled from `N(base_count_time, 0.05 * base_count_time)` using the project's real CSV-backed generator (`E:\Coding Site\just_for_fun\random_no_generation.py`, loaded as `rng` in the first cell) via `rng.random_normal_adv(means, sds)` — one batched draw per section (not per row), since each call does a full read/write of `randomno.csv`'s ~700k-row pool. Returns rounded integers, so durations are integer seconds.
- **`df_general_direct`, non-Swift rows**: `declared_at` = `verified_at` + that row's noisy counting duration.
- **`df_general_proportional`, Swift rows** (6 sub-constituencies, `num_swift_tables` counting tables, currently 3): an N-server FCFS queue starting at `verified_at`. The first `num_swift_tables` sub-constituencies (in CSV order) start immediately, one per table; each subsequent one is dispatched to whichever table frees up first (never waits for more than one table to be free). Each table's duration is independently sampled (batched) with the same noise process.
- **`df_general_direct`, Swift row**: can only declare once every Swift sub-constituency in `df_general_proportional` has -> `declared_at` = max of those 6.
- **`df_general_proportional`, non-Swift rows**: can only declare once every sub-constituency for that Constituency has declared in `df_general_direct` -> `declared_at` = max of `df_general_direct` rows sharing that Constituency.
- **`df_generaldirect_referendum1`**: a constituency's referendum can only declare once that constituency is fully declared in *both* other files -> `declared_at` = max over all `df_general_direct` and `df_general_proportional` rows sharing that Constituency, broadcast to each of that constituency's sub-constituency rows.

In [202]:
import math

extra_ballot_proceed_speed = 2.5
RATE_VOTES_PER_HOUR = 12_000  # 6000 votes / 0.5 hour

def nearest_half_hour(hours):
    """Round UP to the next half hour."""
    return math.ceil(hours * 2) / 2

def base_count_hours(turnout):
    return extra_ballot_proceed_speed * nearest_half_hour(turnout / RATE_VOTES_PER_HOUR)

def noisy_duration_seconds_batch(turnouts):
    """Batched: one randomno.csv round-trip via rng.random_normal_adv for the whole list."""
    means = [base_count_hours(t) * 3600 for t in turnouts]
    sds = [m * 0.05 for m in means]
    return rng.random_normal_adv(means, sds)

general_verified_ts = pd.Timestamp(general_validated_time)

for _df in (df_general_direct, df_general_proportional, df_generaldirect_referendum1):
    _df["received_at"] = general_received_time
    _df["verified_at"] = general_validated_time

In [203]:
# df_general_direct: declared_at for every non-Swift row (Swift resolved later)
_non_swift_mask = df_general_direct["Constituency"] != "Swift"
_non_swift_durations = noisy_duration_seconds_batch(df_general_direct.loc[_non_swift_mask, "Turnout"].tolist())

_declared = pd.Series(
    general_verified_ts + pd.to_timedelta(_non_swift_durations, unit="s"),
    index=df_general_direct.index[_non_swift_mask],
)
df_general_direct["declared_at_dt"] = _declared.reindex(df_general_direct.index)

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


In [ ]:
# df_general_proportional: Swift's 6 sub-constituencies, N-table FCFS queue
num_swift_tables = 3  # EDIT to change number of counting tables

_swift_prop = df_general_proportional[df_general_proportional["Constituency"] == "Swift"]
_swift_durations = noisy_duration_seconds_batch(_swift_prop["Turnout"].tolist())

_table_free = [general_verified_ts] * num_swift_tables
_completion = {}
for (_idx, _row), _dur_s in zip(_swift_prop.iterrows(), _swift_durations):
    _dur = pd.Timedelta(seconds=_dur_s)
    _table = min(range(num_swift_tables), key=lambda t: _table_free[t])
    _end = _table_free[_table] + _dur
    _table_free[_table] = _end
    _completion[_idx] = _end

df_general_proportional["declared_at_dt"] = (
    pd.Series(_completion, dtype="datetime64[ns, UTC]").reindex(df_general_proportional.index)
)

# df_general_direct: Swift row declares once all 6 Swift sub-constituencies in df_general_proportional have
_swift_max = df_general_proportional.loc[
    df_general_proportional["Constituency"] == "Swift", "declared_at_dt"
].max()
df_general_direct.loc[df_general_direct["Constituency"] == "Swift", "declared_at_dt"] = _swift_max

# df_general_proportional: non-Swift rows declare once every sub-constituency for that
# Constituency has declared in df_general_direct
_direct_max_by_const = df_general_direct.groupby("Constituency")["declared_at_dt"].max()
_mask_prop_non_swift = df_general_proportional["Constituency"] != "Swift"
df_general_proportional.loc[_mask_prop_non_swift, "declared_at_dt"] = (
    df_general_proportional.loc[_mask_prop_non_swift, "Constituency"].map(_direct_max_by_const)
)

# df_generaldirect_referendum1: declares once its Constituency is fully declared in BOTH other files
_combined_max = pd.concat([
    df_general_direct[["Constituency", "declared_at_dt"]],
    df_general_proportional[["Constituency", "declared_at_dt"]],
]).groupby("Constituency")["declared_at_dt"].max()

df_generaldirect_referendum1["declared_at_dt"] = (
    df_generaldirect_referendum1["Constituency"].map(_combined_max)
)

# format all three declared_at_dt -> declared_at string, drop the helper column
for _df in (df_general_direct, df_general_proportional, df_generaldirect_referendum1):
    _df["declared_at"] = _df["declared_at_dt"].dt.strftime(TS_FORMAT)
    _df.drop(columns="declared_at_dt", inplace=True)

df_general_direct[["Constituency", "Sub_Constituency", "Turnout", "received_at", "verified_at", "declared_at"]]

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


,Constituency,Sub_Constituency,Turnout,received_at,verified_at,declared_at
0,Planning and Research,Planning and Research,13407,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T15:40:13Z
1,Applied Financial Maths,Financial Risk Management,9676,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T15:02:08Z
2,Applied Financial Maths,Time Series Analysis,9261,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T14:57:29Z
3,Bayesian Modelling and Statistical Machine Lea...,Bayesian Modelling,9834,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T14:50:32Z
4,Bayesian Modelling and Statistical Machine Lea...,Statistical Machine Learning,10512,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T14:58:50Z
5,Coding and Applied Data Science,Mathematics Project,11224,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T15:04:19Z
6,Coding and Applied Data Science,Programming and Data Analysis,10023,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T14:55:06Z
7,Diplomatic Team,Diplomats Rep,6119,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T15:02:49Z
8,Diplomatic Team,Academic and Project Reps,15601,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T16:16:15Z
9,Finance Management,Finance Management,23647,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T17:34:23Z


In [205]:
df_general_proportional

,Constituency,Sub_Constituency,Electorate,Turnout,NRD,LRP,GRF,GCCA,Spoil,received_at,verified_at,declared_at
0,Planning and Research,Planning and Research,14342,13407,5291,977,6753,287,99,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T15:40:13Z
1,Applied Financial Maths,Applied Financial Maths,24000,18937,7852,3537,4763,1598,1187,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T15:02:08Z
2,Bayesian Modelling and Statistical Machine Lea...,Bayesian Modelling and Statistical Machine Lea...,24000,20346,8014,2021,8947,934,430,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T14:58:50Z
3,Coding and Applied Data Science,Coding and Applied Data Science,24000,21247,4806,3100,11894,1135,312,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T15:04:19Z
4,Diplomatic Team,Diplomatic Team,24689,21720,2639,6203,10521,2174,183,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T16:16:15Z
5,Finance Management,Finance Management,27531,23647,2064,8260,3853,9048,422,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T17:34:23Z
6,Decision and Execution,Decision and Execution,9147,8465,690,1920,3727,2035,93,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T14:42:01Z
7,Campaign and Career Team,Campaign and Career Team,17716,16385,5659,2175,7491,803,257,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T15:11:23Z
8,Swift,Swift (Outland),730,631,131,15,62,374,49,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T13:49:13Z
9,Swift,Swift 1,15559,13533,5784,1800,4988,760,201,2026-08-11T02:00:00Z,2026-08-11T12:30:00Z,2026-08-11T16:10:29Z


### df_alma_vale / df_almavale_referendum1

Uses `data/csv_input/df_polygon.csv`, `df_authority.csv`, `df_districts.csv` — plus `df_county.csv`, which turned out to be the missing link for "the Authority a polygon belongs to": `df_polygon.County` -> `df_county.csv` (`County` -> `Zone`) -> `df_authority.csv` (`Authority` == `Zone`). Verified this join is complete (0 unmapped across all 20,850 polygons) and `df_polygon.(District, District_Ward)` matches `df_districts.csv`'s 1,778 ward keys exactly.

All random draws in this section go through `rng.random_normal_adv(means, sds)` (the project's real CSV-backed generator loaded in the first cell), batched once per section across all 20,850 polygons rather than per-row, since each call does a full read/write of `randomno.csv`.

**`received_at`** (ballot transfer, per polygon):
- distance = straight-line distance between the polygon's centroid and its Authority's centroid, converted from map units to km via `8.01 units = 1km`
- `transfer_hours = distance_km / 20` (20 km/h transfer speed)
- sampled from `N(transfer_hours * 1.05, (transfer_hours * 1.05) * 0.075)`, added to `Ballot_Transfter_Time`

**`verified_at`** (per polygon, confirmed rules):
- base duration = `ALMA_VERIFY_SPEED` (1.6) x (`Turnout / ALMA_VERIFY_RATE` (7500)), then rounded **up** to the nearest `alma_round_up_minutes` (currently 15 min; set to 0 to disable rounding entirely) — a separate, finer-grained rule from the general_* section's half-hour ceiling
- polygons are grouped into wards by `(District, District_Ward)`; each ward runs its own independent FCFS queue starting at `alma_vale_verify_time`, in polygon-row order. Wards with more than `large_ward_threshold` (15) polygons get `num_alma_tables_large` (6) tables instead of the default `num_alma_tables` (4)

**`declared_at`** (per polygon, confirmed rules):
- counting for a District can only start once every ward *in that District* has finished verification (barrier = max `verified_at` within the District) — scoped per-District, not a single global barrier
- same per-ward queue mechanism reused for counting (same table-count rule: 6 tables if ward size > 15, else 4), but with `ALMA_COUNT_SPEED` (1.4) and `ALMA_COUNT_RATE` (10000), the same `alma_round_up_minutes` rounding, and starting from the District's barrier time instead of a fixed clock time

**`df_almavale_referendum1`**: copies `received_at` / `verified_at` / `declared_at` from `df_alma_vale` directly, matched by `Shape_ID`.

In [206]:
import ast

poly = pd.read_csv("data/csv_input/df_polygon.csv")
county = pd.read_csv("data/csv_input/df_county.csv")
authority = pd.read_csv("data/csv_input/df_authority.csv")

ALMA_TRANSFER_UNIT_PER_KM = 8.01
ALMA_TRANSFER_SPEED_KMH = 20
Ballot_Transfter_Time = "2026-08-10T22:30:00Z"

def _parse_centroid(s):
    x, y = ast.literal_eval(s)
    return float(x), float(y)

poly[["cx", "cy"]] = poly["centroid"].apply(lambda s: pd.Series(_parse_centroid(s)))
authority[["acx", "acy"]] = authority["centroid"].apply(lambda s: pd.Series(_parse_centroid(s)))

# polygon -> County -> (df_county.csv) Zone -> (df_authority.csv) Authority centroid
poly["Zone"] = poly["County"].map(county.set_index("County")["Zone"])
poly = poly.merge(authority.set_index("Authority")[["acx", "acy"]], left_on="Zone", right_index=True, how="left")
assert poly["acx"].notna().all(), "some polygons could not be mapped to an Authority"

poly["dist_km"] = (((poly["cx"] - poly["acx"]) ** 2 + (poly["cy"] - poly["acy"]) ** 2) ** 0.5) / ALMA_TRANSFER_UNIT_PER_KM
poly["transfer_hours"] = poly["dist_km"] / ALMA_TRANSFER_SPEED_KMH

_transfer_base = pd.Timestamp(Ballot_Transfter_Time)

# batched: one randomno.csv round-trip for all 20,850 polygons instead of one per row
_transfer_means_s = (poly["transfer_hours"] * 1.05 * 3600).tolist()
_transfer_sds_s = [m * 0.075 for m in _transfer_means_s]
_transfer_durations_s = rng.random_normal_adv(_transfer_means_s, _transfer_sds_s)

poly["received_at_dt"] = _transfer_base + pd.to_timedelta(_transfer_durations_s, unit="s")

df_alma_vale = df_alma_vale.merge(
    poly[["Shape_ID", "District", "District_Ward", "received_at_dt"]],
    on="Shape_ID", how="left",
)
assert df_alma_vale["received_at_dt"].notna().all()

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


In [207]:
alma_vale_verify_time = "2026-08-11T10:30:00Z"

ALMA_VERIFY_SPEED = 1.8
ALMA_VERIFY_RATE = 7500
ALMA_COUNT_SPEED = 1.6
ALMA_COUNT_RATE = 5000
num_alma_tables = 2        # EDIT to change number of counting tables per ward
num_alma_tables_large = 5  # EDIT: tables for wards with more than large_ward_threshold polygons
large_ward_threshold = 18
alma_round_up_minutes = 15  # EDIT: round each polygon's base duration up to the nearest N minutes (0 = no rounding)

def nearest_minute_up(hours, minutes):
    if minutes <= 0:
        return hours
    return math.ceil((hours * 60) / minutes) * minutes / 60

def alma_base_hours(turnout, speed, rate):
    base_hours = speed * (turnout / rate)
    return nearest_minute_up(base_hours, alma_round_up_minutes)

def alma_noisy_duration_seconds_batch(turnouts, speed, rate):
    """Batched: one randomno.csv round-trip via rng.random_normal_adv for the whole column."""
    means = [alma_base_hours(t, speed, rate) * 3600 for t in turnouts]
    sds = [m * 0.05 for m in means]
    return rng.random_normal_adv(means, sds)

def tables_for_ward(ward_size):
    return num_alma_tables_large if ward_size > large_ward_threshold else num_alma_tables

def run_ward_queue(group_df, start_times, dur_col, num_tables):
    """FCFS queue with num_tables servers for one ward.
    start_times: a scalar Timestamp, or a Series indexed like group_df giving each row's
    earliest possible start. Returns {row_index: completion_timestamp}."""
    table_free = [None] * num_tables
    completion = {}
    for i, (idx, row) in enumerate(group_df.iterrows()):
        earliest = start_times[idx] if hasattr(start_times, "__getitem__") else start_times
        if i < num_tables:
            table = i
            start = earliest
        else:
            table = min(range(num_tables), key=lambda t: table_free[t])
            start = max(earliest, table_free[table])
        end = start + pd.Timedelta(seconds=row[dur_col])
        table_free[table] = end
        completion[idx] = end
    return completion

In [208]:
# verification: each ward is its own independent FCFS queue starting at alma_vale_verify_time
# (wards with more than large_ward_threshold polygons get num_alma_tables_large tables instead of num_alma_tables)
_verify_start = pd.Timestamp(alma_vale_verify_time)
df_alma_vale["verify_dur_s"] = alma_noisy_duration_seconds_batch(
    df_alma_vale["Turnout"].tolist(), ALMA_VERIFY_SPEED, ALMA_VERIFY_RATE
)

_verify_completion = {}
for _, _g in df_alma_vale.groupby(["District", "District_Ward"], sort=False):
    _verify_completion.update(run_ward_queue(_g, _verify_start, "verify_dur_s", tables_for_ward(len(_g))))

df_alma_vale["verified_at_dt"] = df_alma_vale.index.map(_verify_completion)

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


In [209]:
# counting for a District can only start once every ward IN THAT DISTRICT has finished verification
_district_barrier = df_alma_vale.groupby("District")["verified_at_dt"].transform("max")

df_alma_vale["count_dur_s"] = alma_noisy_duration_seconds_batch(
    df_alma_vale["Turnout"].tolist(), ALMA_COUNT_SPEED, ALMA_COUNT_RATE
)

_count_completion = {}
for _, _g in df_alma_vale.groupby(["District", "District_Ward"], sort=False):
    _starts = _district_barrier.loc[_g.index]
    _count_completion.update(run_ward_queue(_g, _starts, "count_dur_s", tables_for_ward(len(_g))))

df_alma_vale["declared_at_dt"] = df_alma_vale.index.map(_count_completion)

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


In [210]:
df_alma_vale["received_at"] = df_alma_vale["received_at_dt"].dt.strftime(TS_FORMAT)
df_alma_vale["verified_at"] = df_alma_vale["verified_at_dt"].dt.strftime(TS_FORMAT)
df_alma_vale["declared_at"] = df_alma_vale["declared_at_dt"].dt.strftime(TS_FORMAT)

df_alma_vale.drop(columns=[
    "District", "District_Ward", "received_at_dt",
    "verify_dur_s", "verified_at_dt", "count_dur_s", "declared_at_dt",
], inplace=True)
dfs["df_results_alma_vale_pending_verification"] = df_alma_vale

# df_almavale_referendum1: identical received_at / verified_at / declared_at as df_alma_vale
df_almavale_referendum1 = df_almavale_referendum1.drop(columns=["received_at", "verified_at", "declared_at"])
df_almavale_referendum1 = df_almavale_referendum1.merge(
    df_alma_vale[["Shape_ID", "received_at", "verified_at", "declared_at"]],
    on="Shape_ID", how="left",
)
dfs["df_results_almavale_referendum1_pending_verification"] = df_almavale_referendum1

df_alma_vale[["Shape_ID", "Electorate", "Turnout", "received_at", "verified_at", "declared_at"]]

,Shape_ID,Electorate,Turnout,received_at,verified_at,declared_at
0,A01_001,8063,5697,2026-08-10T22:50:04Z,2026-08-11T11:59:46Z,2026-08-11T16:18:47Z
1,A01_002,7316,5148,2026-08-10T22:43:37Z,2026-08-11T11:47:24Z,2026-08-11T16:09:18Z
2,A01_003,3947,2806,2026-08-10T22:45:18Z,2026-08-11T11:11:23Z,2026-08-11T15:23:24Z
3,A01_004,5603,4964,2026-08-10T22:43:05Z,2026-08-11T11:44:02Z,2026-08-11T16:04:13Z
4,A01_005,3935,2967,2026-08-10T22:44:02Z,2026-08-11T11:13:09Z,2026-08-11T15:20:35Z
...,...,...,...,...,...,...
20845,WA06_001,2159,1772,2026-08-10T23:39:21Z,2026-08-11T11:53:11Z,2026-08-11T14:42:15Z
20846,WA07_001,2210,1361,2026-08-11T00:11:28Z,2026-08-11T11:59:34Z,2026-08-11T14:40:10Z
20847,WA08_001,2104,1892,2026-08-10T23:52:37Z,2026-08-11T12:23:03Z,2026-08-11T15:19:57Z
20848,WA09_001,1763,1170,2026-08-10T23:34:30Z,2026-08-11T12:29:46Z,2026-08-11T15:10:40Z


### Alma Vale declaration timing

Earliest/latest polygon to declare, and how many polygons have verified / declared by a few checkpoint times.

In [211]:
# earliest and latest polygon to declare
_declared_dt = pd.to_datetime(df_alma_vale["declared_at"])

earliest_latest_declared = df_alma_vale.loc[
    [_declared_dt.idxmin(), _declared_dt.idxmax()],
    ["Shape_ID", "Electorate", "Turnout", "received_at", "verified_at", "declared_at"],
].copy()
earliest_latest_declared.index = ["earliest", "latest"]
earliest_latest_declared

,Shape_ID,Electorate,Turnout,received_at,verified_at,declared_at
earliest,AQ20_001,1704,1107,2026-08-11T00:15:42Z,2026-08-11T11:00:04Z,2026-08-11T13:12:56Z
latest,H03_001,22157,16100,2026-08-11T00:38:34Z,2026-08-11T14:17:48Z,2026-08-11T19:23:06Z


In [212]:
# how many polygons verified / declared after each checkpoint time
checkpoint_times = ["2026-08-11T16:00:00Z", "2026-08-11T19:00:00Z", "2026-08-11T22:00:00Z"]

_verified_dt = pd.to_datetime(df_alma_vale["verified_at"])

checkpoint_counts = pd.DataFrame({
    "checkpoint": checkpoint_times,
    "verified_after": [(_verified_dt > pd.Timestamp(t)).sum() for t in checkpoint_times],
    "declared_after": [(_declared_dt > pd.Timestamp(t)).sum() for t in checkpoint_times],
}).set_index("checkpoint")

checkpoint_counts

,verified_after,declared_after
checkpoint,,
2026-08-11T16:00:00Z,0,4739
2026-08-11T19:00:00Z,0,4
2026-08-11T22:00:00Z,0,0


In [227]:
df_almavale_referendum1.to_csv("./generated_result/election_results/df_results_almavale_referendum1.csv",index=False)
df_alma_vale.to_csv("./generated_result/election_results/df_results_alma_vale.csv",index=False)
df_generaldirect_referendum1.to_csv("./generated_result/election_results/df_results_generaldirect_referendum1.csv",index=False)
df_general_proportional.to_csv("./generated_result/election_results/df_results_general_proportional.csv",index=False)
df_general_direct.to_csv("./generated_result/election_results/df_results_general_direct.csv",index=False)
df_13D.to_csv("./generated_result/election_results/df_results_13d.csv",index=False)